# Model to analyze replacement of Russian imports by LNG

- no network conversion used
- analysis of the impact of LNG import capacity increasement

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os
from infrastructure_model_data_import_functions import *
from infrastructure_model_input_data_prep_functions import *
from infrastructure_model_output_data_prep_functions import *

### Import Data

In [170]:
# Specify the path to your Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = 'inputs.xlsx'
#data_case_russia_BASE.xlsx
#data_case_russia_peak_hour.xlsx

In [171]:
# Call the function to load the data
df_nodes, df_commodities, df_edges, df_parameter, df_supply_values = load_input_data(input_file_path, excel_file_name)

In [172]:
(network_nodes, commodities, edges, initial_capacities, max_capacities, 
 edge_cost, pipe_new_cost, pipe_conv_cost, pipe_conv_factor, supply_values, 
 node_values) = extract_network_data(
     df_nodes, df_commodities, df_edges, df_parameter, df_supply_values)

### Create input data structure

In [206]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at an infinite (super high) cost
'''
(shortage_list, shortage_edges_list, 
 shortage_capacity_dict, shortage_cost_dict) = create_slack_nodes_and_links(
     node_values, commodities)

In [207]:
shortage_cost_dict

{'Methane': {'shortage_LU_ProdLU_Prod': 100000000,
  'shortage_SE_ProdSE_Prod': 100000000,
  'shortage_NO_ProdNO_Prod': 100000000,
  'shortage_RO_ProdRO_Prod': 100000000,
  'shortage_NL_ProdNL_Prod': 100000000,
  'shortage_AS_ProdAS_Prod': 100000000,
  'shortage_QA_ProdQA_Prod': 100000000,
  'shortage_LV_ProdLV_Prod': 100000000,
  'shortage_TN_ProdTN_Prod': 100000000,
  'shortage_USA_ProdUSA_Prod': 100000000,
  'shortage_EG_ProdEG_Prod': 100000000,
  'shortage_FR_ProdFR_Prod': 100000000,
  'shortage_EL_ProdEL_Prod': 100000000,
  'shortage_SK_ProdSK_Prod': 100000000,
  'shortage_IE_ProdIE_Prod': 100000000,
  'shortage_DZ_ProdDZ_Prod': 100000000,
  'shortage_IM_ProdIM_Prod': 100000000,
  'shortage_UK_ProdUK_Prod': 100000000,
  'shortage_TR_ProdTR_Prod': 100000000,
  'shortage_DK_ProdDK_Prod': 100000000,
  'shortage_CR_ProdCR_Prod': 100000000,
  'shortage_ES_ProdES_Prod': 100000000,
  'shortage_IT_ProdIT_Prod': 100000000,
  'shortage_BE_ProdBE_Prod': 100000000,
  'shortage_EE_ProdEE_Prod'

In [205]:
def create_slack_nodes_and_links(node_values, commodities, methane_value=100000000, hydrogen_value=100000000):
    # Extract all unique node names from node_values (across all commodities)
    all_nodes = set()
    for commodity in node_values.values():
        all_nodes.update(commodity.keys())

    # Filter nodes that contain "_Prod"
    prod_nodes = [node for node in all_nodes if "_Prod" in node]

    # Create a list with names "shortage_" followed by each _Prod node
    shortage_list = [f'shortage_{node}' for node in prod_nodes]

    # **Fix: Flip the direction of edges so that shortage nodes connect to _Prod nodes**
    shortage_edges_list = [(shortage, node) for node, shortage in zip(prod_nodes, shortage_list)]

    # Create a dictionary for capacities of shortage nodes (with infinite capacity represented by a high value)
    shortage_capacity_dict = {
        commodity: {f'{shortage}{node}': 100000000 for node, shortage in zip(prod_nodes, shortage_list)}
        for commodity in commodities
    }

    # Create a dictionary for cost of shortage nodes
    shortage_cost_dict = {
        'Methane': {f'{shortage}{node}': methane_value for node, shortage in zip(prod_nodes, shortage_list)},
        'Hydrogen': {f'{shortage}{node}': hydrogen_value for node, shortage in zip(prod_nodes, shortage_list)}
    }

    # Return the components
    return shortage_list, shortage_edges_list, shortage_capacity_dict, shortage_cost_dict


In [208]:
'''
Create slack nodes for all supply nodes
enable excess nodes that oversupply is also not a problem
'''
(excess_list, excess_edges_list, 
excess_capacity_dict, excess_cost_dict) = create_excess_nodes_for_supply(
    node_values, commodities)

Check the graph

In [209]:
#check that all nodes are connected via edges
check_graph_connectivity(edges)

The graph is connected.


In [210]:
#check that all nodes are connected via edges and identify not connected components
check_graph_connectivity_and_components(edges)

The graph is connected.


In [211]:
# Check if all nodes are connected
check_if_all_nodes_connected(network_nodes, edges)

All nodes are connected.


## Model

### Create model

In [212]:
# Create a new model
model = gp.Model("Grid_Transformation")

### Define parameters

In [213]:
# Parameters
commodities = commodities  # Commodity types
#real network elements
network_nodes = network_nodes # Nodes of the system
network_edges = edges  # Edges
initial_capacities = initial_capacities # Initial capacities
max_capacities = max_capacities  # Maximum capacities
costs_edge = edge_cost  # Cost to transport from node to node
capacity_new_cost = pipe_new_cost  # Cost to increase capacity
capacity_change_cost = pipe_conv_cost  # Cost to increase capacity
node_value = node_values #contains supply and demand values

#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = pipe_conv_factor

#slack parameters shortage
shortage_nodes = shortage_list
shortage_edges = shortage_edges_list
shortage_capacities = shortage_capacity_dict
shortage_cost = shortage_cost_dict

#slack parameters excess
excess_nodes = excess_list
excess_edges = excess_edges_list
excess_capacities = excess_capacity_dict
excess_cost = excess_cost_dict

#complete network of the model
all_edges = network_edges + excess_edges_list + shortage_edges_list

### Define decision variables

In [214]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
x_flow_excess = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    x_flow_excess[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in shortage_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
    for edge in excess_edges:
        x_flow_excess[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_excess_{commodity}_{edge[0]}_{edge[1]}")
model.update()

In [215]:
network_edges_inc_excess = network_edges + excess_edges_list
network_edges_inc_shortage = network_edges + shortage_edges_list

### Define objective and constraints

In [216]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    #gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    #gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges),
    gp.quicksum(x_flow_shortage[commodity][edge] * shortage_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in shortage_edges)+
    gp.quicksum(x_flow_excess[commodity][edge] * excess_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in excess_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges_inc_excess if edge[0] == node)
                                    #substraction of network_edges_inc_shortage not necessary as network_edges_inc_excess
                                    #covers all outgoing flows
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")
#implement excess nodes
for node in excess_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_{commodity}_{node}")

#excess node at the sources to avoid infeasible problems.        
for node in excess_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in excess_edges if edge[1] == node) >= 0,
                        f"supply_excess_{commodity}_{node}")
        
# Add constraint for equality between x_flow and x_flow_excess for the same edge
for commodity in commodities:
    for edge in excess_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_excess[commodity][edge], 
                        f"equality_flow_excess_constraint_{commodity}_{edge}")
        
#Capacity constraint for excess flow
for commodity in commodities:
    for edge in excess_edges:
        model.addConstr(x_flow_excess[commodity][edge] 
                        <= 1000000, 
                        f"capacity_limit_excess{commodity}_{edge[0]}_{edge[1]}")

#implement shortage nodes
for node in shortage_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_shortage_{commodity}_{node}")

#shortage node at the sources to avoid infeasible problems.        
for node in shortage_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in shortage_edges if edge[1] == node) >= 0,
                        f"supply_shortage_{commodity}_{node}")

# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in shortage_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][edge], 
                        f"equality_flow_shortage_constraint_{commodity}_{edge}")

#Capacity constraint for shortage flow
for commodity in commodities:
    for edge in shortage_edges:
        model.addConstr(x_flow_shortage[commodity][edge] 
                        <= 1000000, 
                        f"capacity_limit_shortage{commodity}_{edge[0]}_{edge[1]}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")


# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[commodities[0]][edge] + Change[commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [217]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2606402 - for non-commercial use only - registered to jf___@cbs.dk
Optimize a model with 4632 rows, 3982 columns and 9388 nonzeros
Model fingerprint: 0xa3b4b512
Variable types: 3036 continuous, 946 integer (946 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+07]
  Objective range  [8e+01, 1e+08]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+07]
Presolve removed 4538 rows and 3554 columns
Presolve time: 0.03s
Presolved: 94 rows, 428 columns, 758 nonzeros
Variable types: 428 continuous, 0 integer (0 binary)

Root relaxation: objective 2.615740e+13, 196 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntI

### Results processing

In [218]:
#check model status
check_optimization_status(model)

Optimal solution found!


In [219]:
#call the function to store the results into DataFrames
results_df, excess_df, shortage_df = store_optimization_results(
    model,
    commodities,
    network_edges,
    excess_edges,
    shortage_edges,
    x_flow,
    y_new_cap,
    Change,
    z_conv_cap,
    x_flow_excess,
    x_flow_shortage
)

Get results

In [220]:
# Call the function to filter the DataFrames for each commodity
commodity_dfs = filter_commodities_to_dataframe(results_df, commodities)
methane_rows_df = commodity_dfs.get('Methane')

In [221]:
#create a duplicate for further operations
methane_results_df = methane_rows_df

In [222]:
#get initial capacities but in another format than above
initial_capacities_separator_data = create_initial_capacities_separator_dict(df_parameter)
initial_capacities_data = initial_capacities_separator_data

In [223]:
#get the share of use for each infrastructure element
methane_results_df = add_share_column(methane_results_df, initial_capacities_data)
methane_results_df

,Commodity,Edge,Flow,Share
0,Methane,"(NO, NL)",3.517140e+05,1.000000
1,Methane,"(DZ, ES)",0.000000e+00,0.000000
2,Methane,"(FR, CH)",5.993435e+04,0.736340
3,Methane,"(ES, PT)",0.000000e+00,0.000000
4,Methane,"(MA, ES)",0.000000e+00,0.000000
...,...,...,...,...
369,Methane,"(IN_Prod, IN)",2.321413e+05,0.023214
370,Methane,"(AS_Prod, AS)",1.514322e+06,0.151432
371,Methane,"(CR_Prod, CR)",1.660028e+06,0.166003
372,Methane,"(EG_Prod, EG)",5.712048e+05,0.057120


In [224]:
#aggregated share of capacity use
# Exclude flows and capacities from the following countries
excluded_countries = ['RU']
result_capacity_wo_ru_df = calculate_aggregated_capacity(methane_results_df, initial_capacities_data, excluded_countries)
result_capacity_wo_ru_df

,Commodity,Type,Total Flow,Total Capacity,Share
0,Methane,LNG,1.162189e+07,1.792577e+09,0.006483
1,Methane,Prod,3.157572e+07,5.399999e+08,0.058474
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [225]:
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows from Russia ('RU')
excluded_countries = ['RU']
result_supply_wo_ru_df = calculate_aggregated_share_supply(methane_results_df, node_values, excluded_countries)
result_supply_wo_ru_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,1.162189e+07,0.000000e+00,0.000000
1,Methane,Prod,3.157572e+07,3.158367e+07,0.999748
2,Methane,St,0.000000e+00,0.000000e+00,0.000000


In [226]:
search_element = 'RU'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element_and_commodity(results_df, 'Edge', search_element, commodity)
filtered_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity
17,Methane,"(LT, RU)",0.000000e+00,0.0,1.0,41683.0
31,Methane,"(RU, LV)",0.000000e+00,0.0,1.0,61320.0
52,Methane,"(LV, RU)",0.000000e+00,0.0,1.0,31025.0
58,Methane,"(RU, DE)",0.000000e+00,0.0,1.0,513190.0
70,Methane,"(RU, UA)",0.000000e+00,0.0,1.0,486910.0
75,Methane,"(RU, BY)",0.000000e+00,0.0,1.0,439277.5
89,Methane,"(RU, FI)",0.000000e+00,0.0,1.0,80300.0
98,Methane,"(RU, TR)",0.000000e+00,0.0,1.0,349998.5
132,Methane,"(RU, RU_LNG)",1.728931e+06,0.0,1.0,9999999.0
142,Methane,"(RU, CN)",3.712600e+05,0.0,1.0,371260.0


In [227]:
search_element = 'USA'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element(methane_results_df, 'Edge', search_element)
filtered_df

,Commodity,Edge,Flow,Share
133,Methane,"(USA, USA_LNG)",8.193730e+05,0.081937
137,Methane,"(USA, CM)",6.249942e+04,0.012794
139,Methane,"(CM, USA)",0.000000e+00,0.000000
157,Methane,"(USA_LNG, EL_LNG)",0.000000e+00,0.000000
173,Methane,"(USA_LNG, JS_LNG)",0.000000e+00,0.000000
181,Methane,"(USA_LNG, SA_LNG)",0.000000e+00,0.000000
198,Methane,"(USA_LNG, PL_LNG)",0.000000e+00,0.000000
201,Methane,"(USA_LNG, IT_LNG)",0.000000e+00,0.000000
202,Methane,"(USA_LNG, EE_LNG)",2.444313e+04,0.002444
203,Methane,"(USA_LNG, ES_LNG)",0.000000e+00,0.000000


In [228]:
#get just excess for methane out of the excess_df
methane_excess_df = excess_df[excess_df['Commodity'].str.contains('Methane', case=False)]

In [229]:
'''still not working'''
#check for an unused share of capacities
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows
excluded_countries = ['ES']
methane_excess_share_df = calculate_aggregated_share_supply(methane_excess_df, node_values, excluded_countries)
methane_excess_share_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,0.000000,0.000000e+00,0.000000
1,Methane,Prod,261057.941137,3.782085e+07,0.006902
2,Methane,St,0.000000,0.000000e+00,0.000000


In [230]:
shortage_df

methane_shortage_df = shortage_df[shortage_df['Commodity'].str.contains('Methane', case=False)]
methane_shortage_df

,Commodity,Edge,Flow
0,Methane,"(shortage_SK_Prod, SK_Prod)",588.271902
1,Methane,"(shortage_MD_Prod, MD_Prod)",1117.368710
2,Methane,"(shortage_UA_Prod, UA_Prod)",69708.985980
3,Methane,"(shortage_PL_Prod, PL_Prod)",5357.041567
4,Methane,"(shortage_BA_Prod, BA_Prod)",2907.090436
5,Methane,"(shortage_BY_Prod, BY_Prod)",173427.356460


In [231]:
methane_results_df.to_excel("output1.xlsx")